In [77]:
import os
import h5py
import xarray as xr
import glob
import pandas as pd

os.chdir("/Users/vincentribberink/Documents/School/Winter 2026/GEOM 4009/Final Project/")

os.listdir()

['Sample Data',
 '.DS_Store',
 'peat_vector',
 '.ipynb_checkpoints',
 'ICESat-2_h5_to_xr.ipynb',
 'GEDI Example.ipynb']

In [76]:
### Xarray option


# import icesat-2 data
test_files = glob.glob("Sample Data/ATL08_007-20260210_184835/*.h5")

# list of groups containing beam data
beam_list = ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"] # groups in h5 file

# create empty list #1
ds_combined_list = []

# iterate through h5 files in directory
for file in test_files:
    # read file with H5py
    f = h5py.File(file)
    
    # create empty list #2
    ds_list = []

    # iterate through groups to find beam data
    for group in list(f.keys()):
        if group in beam_list:
            
            # create xarray dataset with canopy height, coordinates
            ds = xr.Dataset(
                data_vars = dict(
                    h_canopy=("segment", f[f"{group}/land_segments/canopy/h_canopy"][:])
                    ),
                coords = dict(
                    lon=("segment", f[f"{group}/land_segments/longitude"][:]),
                    lat=("segment", f[f"{group}/land_segments/latitude"][:])
                    ),
                attrs={"beam": group} # beam type
                )

            # add to list #2
            ds_list.append(ds)

    # concatenate all beams for a given file
    ds_combined = xr.concat(ds_list, dim="segment")
    
    # add to list #1
    ds_combined_list.append(ds_combined)

# concatenate all files
final = xr.concat(ds_combined_list, dim="segment")

In [87]:
### Pandas Option

# import icesat-2 data
test_files = glob.glob("Sample Data/ATL08_007-20260210_184835/*.h5")

# list of groups containing beam data
beam_list = ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"] # groups in h5 file

# create empty list #1
df_combined_list = []

# iterate through h5 files in directory
for file in test_files:
    # read file with H5py
    f = h5py.File(file)
    
    # create empty list #2
    df_list = []

    # iterate through groups to find beam data
    for group in list(f.keys()):
        if group in beam_list:
            
            # create pandas dataframe with canopy height, coordinates
            df = pd.DataFrame({
                    "h_canopy" : f[f"{group}/land_segments/canopy/h_canopy"][:],
                    "longitude" : f[f"{group}/land_segments/longitude"][:],
                    "latitude" : f[f"{group}/land_segments/latitude"][:],
                    "beam" : str(group)
            })
            
            # add to list #2
            df_list.append(df)

    # concatenate all beams for a given file
    df_combined = pd.concat(df_list)
    
    # add to list #1
    df_combined_list.append(df_combined)

# concatenate all files
final = pd.concat(df_combined_list).reset_index()

# export
final.to_csv("icesat_sample.csv")

In [38]:
# this method might be easier, but I cannot get it to work
import icepyx as ipx

sample_data_dir = glob.glob("Sample Data/ATL08_007-20260210_184835/*.h5")

reader = ipx.Read(data_source=sample_data_dir)
reader.variables.append(beam_list=['gt1l', 'gt1r', 'gt2l', 'gt2r', 'gt3l', 'gt3r'], 
                        var_list=['h_canopy', 'latitude', 'longitude'])
ds=reader.load()

ValueError: Dimension photon_idx already exists.